# D1 feature-availability audit (validation window only)

Scores the **frozen** models (1h/6h LightGBM, 24h hybrid) on Sep 1 - Oct 31 2025 with live-source gaps applied:
no solar radiation and no ITO weather (OpenAQ primary), or no NOx and no station weather (data.gov.in primary),
each with the gaps filled from Open-Meteo/ERA5 or left empty. Nothing is trained. The test window is never read.

**Before running:** upload `tier3_d1_audit_bundle.zip` to your Drive folder (or to `/content`), then run all cells.
The notebook stops if any frozen model's hash differs from the one recorded when this notebook was generated.

In [ ]:
import os, sys, hashlib, subprocess, zipfile, pathlib, platform
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = pathlib.Path('/content/drive/MyDrive/aq_tier3_audit')
except ImportError:                      # running locally
    PROJECT_DIR = pathlib.Path('.').resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE = next((p for p in [PROJECT_DIR / 'tier3_d1_audit_bundle.zip', pathlib.Path('/content/tier3_d1_audit_bundle.zip')] if p.exists()), None)
if BUNDLE is not None:
    zipfile.ZipFile(BUNDLE).extractall(PROJECT_DIR)
    print('extracted', BUNDLE)
os.chdir(PROJECT_DIR); sys.path.insert(0, str(PROJECT_DIR))
print('project dir:', PROJECT_DIR)

In [ ]:
EXPECTED_SHA256 = {
    "models/lgb_model_1h_v1.joblib": "90c9167b22128f23cce46d51a7b163164c1022f25375fadae3b1fa08d0945095",
    "models/lgb_model_6h_v1.joblib": "a0305bd29b496e7fe64a24fb336558f6883e65c4bb5e6ec1159fc5cd2c47e17f",
    "models/24h/final/24h_final_model.joblib": "b2069a4b5671cf1ec7e4047922c351514513d5f2b8ccf4256377b5a03bc829f8",
    "models/24h/final/scaler_final_24h.joblib": "bf2ea17accc327515ceb1f499dac38ae49a232bfd80ea299a5b340bbe9ff74ea",
    "models/24h/final/imputer_final_24h.joblib": "3ee82708a96a30ccc1da28fe537084eb0659a3c24a2def6ea2eeea855af788c4",
    "models/24h/final/feature_names_final.joblib": "bf11b205a39bcb994b87bb88b47c46df7b794ae4aaafc795b2f826251b84d00b"
}
def sha(path): return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()
bad = [f for f, h in EXPECTED_SHA256.items() if not pathlib.Path(f).exists() or sha(f) != h]
assert not bad, f'frozen model files missing or different from the recorded hashes: {bad}'
missing = [f for f in ["data/processed/features_extended.parquet", "data/processed/exogenous_weather.parquet"] if not pathlib.Path(f).exists()]
assert not missing, f'missing data files: {missing}'
assert list(pathlib.Path('Og Data/Delhi data').glob('*/Raw Data/2025_*_1H.csv')), 'raw 2025 CSVs missing'
print('frozen models verified against recorded hashes; inputs present')

In [ ]:
# lightgbm is needed for the 1h/6h models; psycopg2 is imported (not used) by the shared feature module
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'psycopg2-binary', 'pyarrow', 'python-dotenv', 'requests'], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
import sklearn, lightgbm, pandas, numpy
print('python', platform.python_version(), '| sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__)
print('NOTE: the frozen models were pickled with other library versions; load warnings below are expected, a crash is not.')

In [ ]:
r = subprocess.run([sys.executable, '-W', 'ignore', '-m', 'src.evaluation.feature_availability_audit', '--horizons', '24', '1', '6'], capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:])
assert r.returncode == 0, 'the audit failed; do not use partial output'

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
out = pathlib.Path('reports/modeling/live_audit')
display(Markdown((out / 'feature_availability_audit.md').read_text()))
table = pd.read_csv(out / 'feature_availability_audit.csv')
assert set(table.horizon) == {'1h', '6h', '24h'}, f'horizons present: {sorted(set(table.horizon))}'
table[['horizon', 'scenario', 'n', 'mae', 'mae_persistence', 'skill_mae', 'delta_mae_vs_baseline', 'delta_mae_ci_lo', 'delta_mae_ci_hi']].round(3)

In [ ]:
import shutil
shutil.make_archive('/content/d1_audit_results' if os.path.exists('/content') else 'd1_audit_results', 'zip', 'reports/modeling/live_audit')
try:
    from google.colab import files; files.download('/content/d1_audit_results.zip')
except ImportError:
    print('results zip written locally')